# What is the classification compared with?

Separate training from validation regions and examine the reference data.

**Task** · Train in the west, validate in the east and interpret the confusion matrix.

Run from top to bottom. Predict the result before changing code, then compare it with the output.

## Before you start

Use a CPU runtime. You need a registered Earth Engine Cloud project and permission to use it. See the [setup guide](https://github.com/steelpl/lecture/blob/main/CE776-climate-change-carbon-water-cycle/GEE_SETUP.md). Authenticate with your own account before querying data.

In [ ]:
%pip -q install earthengine-api==1.7.52 geemap==0.39.0

In [ ]:
import ee
import geemap

PROJECT_ID = ""  # Your Cloud project ID registered for Earth Engine
if not PROJECT_ID.strip():
    raise ValueError("Enter your registered Google Cloud project ID in PROJECT_ID")
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

In [ ]:
roi = ee.Geometry.Rectangle([126.80, 37.43, 127.15, 37.70], geodesic=False)

In [ ]:
def mask_s2(image):
    scl = image.select('SCL')
    clear = (scl.neq(1).And(scl.neq(3)).And(scl.neq(8))
             .And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11)))
    return (image.select(['B2', 'B3', 'B4', 'B8', 'B11'])
            .multiply(0.0001).updateMask(clear)
            .copyProperties(image, ['system:time_start']))
collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
              .filterBounds(roi).filterDate('2020-05-01', '2020-10-01')
              .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 60))
              .map(mask_s2))

image = collection.median().clip(roi)
bands = ['B2','B3','B4','B8','B11']
wc = ee.ImageCollection('ESA/WorldCover/v100').first().select('Map')
classes = [10,20,30,40,50,60,70,80,90,95,100]
labels = wc.remap(classes, list(range(11))).rename('label').toByte()
stack = image.select(bands).addBands(labels)
west = ee.Geometry.Rectangle([126.80,37.43,126.94,37.70], geodesic=False)
east = ee.Geometry.Rectangle([127.01,37.43,127.15,37.70], geodesic=False)
training = stack.stratifiedSample(numPoints=60,classBand='label',region=west,scale=30,seed=7,geometries=False)
validation = stack.stratifiedSample(numPoints=40,classBand='label',region=east,scale=30,seed=8,geometries=False)
model = ee.Classifier.smileRandomForest(numberOfTrees=50,seed=7).train(training,'label',bands)
classified = image.select(bands).classify(model)
matrix = validation.classify(model).errorMatrix('label','classification',ee.List.sequence(0,10))

In [ ]:
print('train / validation:', training.size().getInfo(), validation.size().getInfo())
print('confusion matrix:', matrix.getInfo())
print('class-stratified sample agreement:', matrix.accuracy().getInfo())

In [ ]:
Map = geemap.Map()
Map.centerObject(roi, 10)
palette = ['006400','ffbb22','ffff4c','f096ff','fa0000','b4b4b4','f0f0f0','0064c8','0096a0','00cf75','fae6a0']
Map.addLayer(image, {'bands':['B4','B3','B2'],'min':0,'max':0.3}, 'RGB')
Map.addLayer(labels, {'min':0,'max':10,'palette':palette}, 'WorldCover reference')
Map.addLayer(classified, {'min':0,'max':10,'palette':palette}, 'RF classes')
Map.addLayer(west, {'color':'blue'}, 'Training region')
Map.addLayer(east, {'color':'red'}, 'Validation region')
Map

Locate the training and validation regions on the map. Identify commonly confused classes and compare them with the RGB image.

**Scope** · WorldCover is a reference map derived from related satellite observations, not independent field truth. Report these scores as reference-map agreement. Class-stratified sampling means simple overall agreement is not an area-weighted regional accuracy estimate. Spatial separation does not guarantee removal of every dependence.

Matrix row/column order: tree cover, shrubland, grassland, cropland, built-up, bare/sparse vegetation, snow/ice, water, herbaceous wetland, mangroves, moss/lichen. The reference and RF maps use the same colors.

## Data and interpretation

- [Sentinel-2 SR](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED)

- [WorldCover 2020](https://developers.google.com/earth-engine/datasets/catalog/ESA_WorldCover_v100)

Observation years define fixed examples, independent of the teaching year. The code demonstrates processing methods; inspect actual maps and values after authentication.